# Small Wind Turbine for Modena  Day 1 starter notebook

**Goal:** estimate how much electricity a small wind turbine could realistically
produce in Modena, Italy, and find the design that gets the most out of the local wind.

This notebook already runs. Press **Runtime  Run all** (in Google Colab) and read
down the page. Then start changing the numbers in the **INPUTS** cell and watch the
results move. Understanding *why* they move is the whole point of the project.

---
### Day-1 task: get your own real numbers
The values below are good Modena estimates, but part of showing initiative is pulling
the real data yourself and citing it:
1. Open **globalwindatlas.info**, search *Modena*, click the map. Read the **mean wind
   speed** and the **Weibull A and k** values at 10 m, 50 m and 100 m. Write them down.
2. Cross-check on **power.larc.nasa.gov/data-access-viewer** (NASA POWER): enter Modena's
   coordinates (44.65 N, 10.93 E) and look at WS10M / WS50M.
3. Replace the numbers in the INPUTS cell with what you found, and note your source.


## 1. Inputs  everything you can change is here
Modena is in the Po Valley, one of the lowest-wind regions in Europe (surface winds
average roughly **2 m/s** at 10 m). We capture that with a **Weibull distribution**, the
standard way engineers describe how often each wind speed occurs at a site.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma

# ---- SITE: Modena, Po Valley (replace with your Global Wind Atlas numbers) ----
A10   = 2.25   # Weibull scale parameter at 10 m height [m/s]  (~ mean wind speed)
k     = 1.8    # Weibull shape parameter  (lower = more variable wind)
alpha = 0.25   # wind-shear exponent: how fast wind grows with height (suburban land)

# ---- TURBINE DESIGN (these are the choices you will optimise later) ----
D        = 3.0    # rotor diameter [m]
hub      = 18.0   # tower / hub height [m]
Cp       = 0.35   # power coefficient: fraction of wind energy captured (max = 0.593)
v_in     = 3.0    # cut-in wind speed: turbine starts spinning [m/s]
v_rated  = 11.0   # rated wind speed: turbine hits full power [m/s]
v_out    = 25.0   # cut-out wind speed: turbine shuts down for safety [m/s]

# ---- CONSTANTS / LOCAL CONTEXT ----
rho        = 1.225   # air density [kg/m^3] at sea level
HOURS_YEAR = 8760    # hours in a year
HOME_KWH   = 2700    # avg Italian household electricity use [kWh/yr] (ARERA standard)
CO2_KG_KWH = 0.31    # Italy grid carbon intensity [kg CO2 / kWh] (~310 g, 2024)

print("Inputs loaded. Modena Weibull: A =", A10, "m/s, k =", k)


## 2. Layer 1  how much power is in the wind
The core equation of wind energy:
$$P = \tfrac{1}{2}\,\rho\,A\,v^{3}\,C_p$$
The **cube on v** is the key insight: doubling wind speed gives 8x the power. That single
fact is why tower height and site choice matter so much.


In [ ]:
r = D / 2
swept_area = np.pi * r**2          # area the blades sweep [m^2]
P_rated = 0.5 * rho * swept_area * v_rated**3 * Cp   # full-power output [W]

def turbine_power(v):
    """Power output [W] of the turbine at wind speed v [m/s]."""
    v = np.asarray(v, dtype=float)
    P = np.zeros_like(v)
    ramp = (v >= v_in) & (v < v_rated)      # building up to full power
    P[ramp] = 0.5 * rho * swept_area * v[ramp]**3 * Cp
    flat = (v >= v_rated) & (v < v_out)     # capped at rated power
    P[flat] = P_rated
    return P                                 # 0 below cut-in and above cut-out

v = np.linspace(0, 30, 601)
plt.figure(figsize=(7,4))
plt.plot(v, turbine_power(v)/1000, lw=2)
plt.axvline(v_in, ls='--', c='green', label=f'cut-in {v_in} m/s')
plt.axvline(v_rated, ls='--', c='orange', label=f'rated {v_rated} m/s')
plt.title(f'Power curve  {D} m rotor, Cp={Cp}  (rated {P_rated/1000:.2f} kW)')
plt.xlabel('wind speed [m/s]'); plt.ylabel('power [kW]')
plt.legend(); plt.grid(alpha=.3); plt.show()


## 3. Layer 2  Modena's wind is rarely strong
A turbine only makes power when the wind blows. The Weibull curve below shows how often
each wind speed happens in Modena. Notice how much of it sits **left of the green cut-in
line**  that wind is too weak to turn the blades at all.


In [ ]:
def weibull_pdf(v, A, k):
    """Probability density of wind speed v for a Weibull(A, k) site."""
    v = np.asarray(v, dtype=float)
    return (k/A) * (v/A)**(k-1) * np.exp(-(v/A)**k)

# scale the wind up to hub height using the wind-shear power law:  v(h) = v(10) * (h/10)^alpha
A_hub = A10 * (hub/10.0)**alpha
mean_10  = A10   * gamma(1 + 1/k)
mean_hub = A_hub * gamma(1 + 1/k)

plt.figure(figsize=(7,4))
plt.plot(v, weibull_pdf(v, A10,   k), lw=2, label=f'at 10 m  (mean {mean_10:.1f} m/s)')
plt.plot(v, weibull_pdf(v, A_hub, k), lw=2, label=f'at {hub:.0f} m hub (mean {mean_hub:.1f} m/s)')
plt.axvline(v_in, ls='--', c='green', label=f'cut-in {v_in} m/s')
plt.title('How often each wind speed occurs in Modena')
plt.xlabel('wind speed [m/s]'); plt.ylabel('probability density')
plt.legend(); plt.grid(alpha=.3); plt.show()

frac_above_cutin = np.exp(-(v_in/A_hub)**k)   # Weibull survival function
print(f'Wind is above cut-in only {frac_above_cutin*100:.0f}% of the time at {hub:.0f} m.')


## 4. Layer 3  annual energy, the number that actually matters
Multiply the power curve by how often each wind speed happens, across all 8760 hours in a
year. That gives **Annual Energy Production (AEP)** in kWh  then we compare it to a typical
Italian home and convert to CO2 avoided.


In [ ]:
def annual_energy_kwh(A_scale, k):
    """Expected yearly energy [kWh] given a Weibull(A_scale, k) wind resource."""
    vv = np.linspace(0, 30, 1201)
    pdf = weibull_pdf(vv, A_scale, k)
    mean_power_W = np.trapz(turbine_power(vv) * pdf, vv)   # expected power [W]
    return mean_power_W * HOURS_YEAR / 1000.0

AEP = annual_energy_kwh(A_hub, k)
capacity_factor = (AEP*1000/HOURS_YEAR) / P_rated
offset_pct = AEP / HOME_KWH * 100
co2_saved  = AEP * CO2_KG_KWH

print('================  MODENA RESULT  ================')
print(f'Annual energy produced : {AEP:6.0f} kWh/year')
print(f'Capacity factor        : {capacity_factor*100:5.1f} %   (how hard the turbine works)')
print(f'Share of one home       : {offset_pct:5.1f} %   (of {HOME_KWH} kWh/yr)')
print(f'CO2 avoided            : {co2_saved:6.0f} kg/year')
print('=================================================')


## 5. First optimisation views  what helps most?
Two design knobs: how **big** the rotor is, and how **tall** the tower is. Because power
goes with diameter squared and with wind speed cubed, both matter  but in Modena's weak
wind, see which one actually moves the needle.


In [ ]:
# --- AEP vs rotor diameter (tower height fixed) ---
Ds = np.linspace(1, 6, 30)
aep_D = []
for d in Ds:
    r_ = d/2; area_ = np.pi*r_**2; Prated_ = 0.5*rho*area_*v_rated**3*Cp
    def Pf(v, area_=area_, Prated_=Prated_):
        v=np.asarray(v,float); P=np.zeros_like(v)
        ramp=(v>=v_in)&(v<v_rated); P[ramp]=0.5*rho*area_*v[ramp]**3*Cp
        flat=(v>=v_rated)&(v<v_out); P[flat]=Prated_
        return P
    vv=np.linspace(0,30,1201); pdf=weibull_pdf(vv,A_hub,k)
    aep_D.append(np.trapz(Pf(vv)*pdf,vv)*HOURS_YEAR/1000)

# --- AEP vs hub height (rotor diameter fixed) ---
Hs = np.linspace(10, 30, 30)
aep_H = [annual_energy_kwh(A10*(h/10.0)**alpha, k) for h in Hs]

fig, ax = plt.subplots(1, 2, figsize=(11,4))
ax[0].plot(Ds, aep_D, lw=2, color='C0'); ax[0].set_title('AEP vs rotor diameter')
ax[0].set_xlabel('rotor diameter [m]'); ax[0].set_ylabel('kWh/year'); ax[0].grid(alpha=.3)
ax[1].plot(Hs, aep_H, lw=2, color='C2'); ax[1].set_title('AEP vs tower height')
ax[1].set_xlabel('hub height [m]'); ax[1].set_ylabel('kWh/year'); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. Reality check  Modena vs a genuinely windy site
Same turbine, two locations: Modena versus a good coastal/ridge site (mean ~6.5 m/s). This
contrast is the honest, interesting story for your write-up  a good engineer reports when
a site *isn't* suitable, and explains why.


In [ ]:
A_good = 7.33   # Weibull scale for a windy site, mean ~6.5 m/s
aep_modena = annual_energy_kwh(A_hub, k)
aep_good   = annual_energy_kwh(A_good*(hub/10.0)**alpha, 2.0)

plt.figure(figsize=(6,4))
bars = plt.bar(['Modena\n(Po Valley)', 'Windy site\n(~6.5 m/s)'],
               [aep_modena, aep_good], color=['#c0504d','#4f81bd'])
for b,val in zip(bars,[aep_modena,aep_good]):
    plt.text(b.get_x()+b.get_width()/2, val, f'{val:.0f}', ha='center', va='bottom')
plt.ylabel('Annual energy [kWh/year]'); plt.title('Why site choice dominates')
plt.grid(axis='y', alpha=.3); plt.show()

print(f'The same turbine makes ~{aep_good/max(aep_modena,1):.0f}x more energy at the windy site.')
print('Takeaway: in Modena the limiting factor is the wind resource, not the turbine.')


## What you have now
A working model that turns *site wind + turbine design* into *kWh/year, home-offset %, and
CO2 saved*  for your own city.

**Day 2:** replace the inputs with your real Global Wind Atlas numbers, then start the
optimisation in earnest  sweep diameter, hub height and Cp together to find Modena's best
realistic design (and decide honestly whether wind even makes sense here versus solar).
Ask me and I'll walk you through each step.
